In [1]:
import pandas as pd
from pathlib import Path

# Chargement du jeu brut
DATA_DIR = Path("data")
train = pd.read_csv(DATA_DIR / "train.csv")

In [2]:
TITLE_MAP = {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"}
COMMON_TITLES = ["Mr", "Mrs", "Miss", "Master"]


def extract_title(name):
    title = name.str.extract(r",\s*([^.]+)\.", expand=False).str.strip()
    title = title.replace(TITLE_MAP)
    return title.where(title.isin(COMMON_TITLES), "Rare")


# Titres de train pour calculer l'âge médian par titre
train_title = extract_title(train["Name"])
# Âge médian par titre
age_by_title = train.groupby(train_title)["Age"].median()
# Âge médian global (secours si un titre est inconnu)
age_median = train["Age"].median()
# Port le plus fréquent
embarked_mode = train["Embarked"].mode()[0]
# Prix médian par classe
fare_by_pclass = train.groupby("Pclass")["Fare"].median()



def clean(df, age_by_title, age_median, embarked_mode, fare_by_pclass):
    df = df.copy()  # ne modifie pas le DataFrame d'origine
    # Nouvelle variable : le titre
    df["Title"] = extract_title(df["Name"])
    # Age manquant -> médiane du titre (sinon médiane globale)
    df["Age"] = df["Age"].fillna(df["Title"].map(age_by_title)).fillna(age_median)
    # Embarked manquant -> port le plus fréquent
    df["Embarked"] = df["Embarked"].fillna(embarked_mode)
    # Fare manquant -> prix médian de la classe
    df["Fare"] = df["Fare"].fillna(df["Pclass"].map(fare_by_pclass))
    # Taille de la famille (le passager compte) et indicateur voyage seul
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    # Retire Name (remplacé par Title), Ticket (inutile) et Cabin (trop vide)
    return df.drop(columns=["Name", "Ticket", "Cabin"])

In [3]:
train_clean = clean(train, age_by_title, age_median, embarked_mode, fare_by_pclass).drop(columns="PassengerId")

In [4]:
train_clean.isna().sum().sum()

np.int64(0)

In [5]:
train_clean.head(20)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,FamilySize,IsAlone
0,0,3,male,22.0,1,0,7.2500,S,Mr,2,0
1,1,1,female,38.0,1,0,71.2833,C,Mrs,2,0
2,1,3,female,26.0,0,0,7.9250,S,Miss,1,1
3,1,1,female,35.0,1,0,53.1000,S,Mrs,2,0
4,0,3,male,35.0,0,0,8.0500,S,Mr,1,1
5,0,3,male,30.0,0,0,8.4583,Q,Mr,1,1
6,0,1,male,54.0,0,0,51.8625,S,Mr,1,1
7,0,3,male,2.0,3,1,21.0750,S,Master,5,0
8,1,3,female,27.0,0,2,11.1333,S,Mrs,3,0
9,1,2,female,14.0,1,0,30.0708,C,Mrs,2,0


In [6]:
train_clean.to_csv(DATA_DIR / "train_clean.csv", index=False)